<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Relational concepts

**[Relational concepts](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/)** · SQL and Data Preparation for AI · Module 1, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** connect tables with primary and foreign keys, read a one-to-many relationship, and check the keys of a real export. The export has 14 duplicated ticket IDs and rows that point to nothing.

| | |
|---|---|
| **Time** | About 50 minutes |
| **Needs** | An internet connection for the first cell. No account and no install: only Python's standard library, and pandas to show tables if it is there (it is in Colab and Kaggle). |
| **You should know** | Rows, columns, NULL and the grain of each table, and `SELECT … FROM … LIMIT`, from [Dataset anatomy](https://learning.nextia-ai.com/courses/sql/m01/dataset-anatomy/). |
| **You do not need** | Anything on your computer. The setup below builds the practice database here. |
| **Tested** | Python 3.14.6 (SQLite 3.50.4), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/) has the full explanations, the entity relationship diagram and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your query between the triple quotes `"""`, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M01-L02-relational-concepts/relational-concepts-solution.ipynb).

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Three small tables

The first part of this lesson uses three small tables, so that you can see every row: 3 customers, 5 tickets and 5 outcome events. The next cell makes them in memory, as `tiny`. The practice database `con` does not change.

Run the cell. It prints nothing.

In [ ]:
tiny = sqlite3.connect(":memory:")
tiny.executescript("""
CREATE TABLE customers (customer_id TEXT, segment TEXT);
INSERT INTO customers VALUES
    ('C-0001', 'home'), ('C-0002', 'trade'), ('C-0003', 'business');
CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, team TEXT);
INSERT INTO tickets VALUES
    ('T-20001', 'C-0001', 'delivery'), ('T-20002', 'C-0002', 'returns'),
    ('T-20003', 'C-0002', 'payment'), ('T-20004', 'C-0002', 'delivery'),
    ('T-20005', 'C-0001', 'account');
CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, status TEXT);
INSERT INTO outcomes VALUES
    ('O-90001', 'T-20001', 'resolved'), ('O-90002', 'T-20002', 'escalated'),
    ('O-90003', 'T-20002', 'resolved'), ('O-90004', 'T-20003', 'resolved'),
    ('O-90005', 'T-20005', 'resolved');
""");

Run the next cell to see the tickets. Then change `tickets` to `outcomes`, and to `customers`, and run it each time.

In [ ]:
sql("""
SELECT *
FROM tickets;
""", db=tiny)

> **Check.** You should see 5 tickets, T-20001 to T-20005. `outcomes` has 5 events, and `customers` has 3 customers.

## 2. Keys

- A **primary key** is a column whose value is different in every row and never NULL. It identifies one row. In `customers`, it is `customer_id`. In `tickets`, it is `ticket_id`, and in `outcomes`, `outcome_id`.
- A **foreign key** is a column whose values point to the primary key of another table. `tickets.customer_id` points to `customers.customer_id`, and `outcomes.ticket_id` points to `tickets.ticket_id`.

`customers.customer_id` means "the column `customer_id` of the table `customers`".

| | Primary key | Foreign key |
|---|---|---|
| Can a value appear twice? | No | Yes: one customer has many tickets |
| Can it be NULL? | No | Yes, if "no match" is allowed (a guest ticket) |
| What must be true? | Unique and not NULL | Each value that is not NULL exists in the other table |

## 3. One-to-many

One customer has zero or more tickets, and each ticket has one customer. This is a **one-to-many** relationship.

> **Predict.** How many tickets does C-0002 have? Then run the cell.

In [ ]:
sql("""
SELECT *
FROM tickets
WHERE customer_id = 'C-0002';
""", db=tiny)

> **Check.** You should see 3 rows: T-20002, T-20003 and T-20004.

> **Your turn: change one thing.** Change the query so that it shows the events of ticket T-20002 from `outcomes`, on the tiny tables. Then run the check cell.

In [ ]:
events_t20002 = """
SELECT *
FROM tickets
WHERE customer_id = 'C-0002';
"""
sql(events_t20002, db=tiny)

In [ ]:
check(events_t20002, rows=2, columns=3, expected="81e3a466c53f4c94", db=tiny)

T-20002 has two events: escalated, then resolved. T-20004 has none yet. The foreign key is always on the "many" side: each ticket names its one customer, and each event names its one ticket.

The [entity relationship diagram](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/#the-entity-relationship-diagram) on the lesson page draws the three tables of the practice database, their keys and these relationships.

## 4. What a constraint does

A **constraint** is a rule in the schema that the database checks on every change. `PRIMARY KEY` rejects a second row with the same value. `FOREIGN KEY` rejects a value that does not exist in the other table.

The practice database `con` is read-only, so the next cell makes a new database in memory, `rules`. It has two tables with constraints, two customers (C-0001 and C-0002) and one ticket, T-20001. It also defines `try_sql()`, which runs SQL and prints `Accepted.` or the error. Run the cell.

In [ ]:
def try_sql(script, db):
    """Run SQL statements. Print "Accepted." or the error from the database."""
    try:
        db.executescript(script)
        print("Accepted.")
    except sqlite3.Error as error:
        print("Error:", error)

rules = sqlite3.connect(":memory:")
try_sql("""
PRAGMA foreign_keys = ON;
CREATE TABLE customers (
    customer_id TEXT NOT NULL PRIMARY KEY,
    segment     TEXT
);
CREATE TABLE tickets (
    ticket_id   TEXT NOT NULL PRIMARY KEY,
    customer_id TEXT,
    team        TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);
INSERT INTO customers VALUES ('C-0001', 'home'), ('C-0002', 'trade');
INSERT INTO tickets VALUES ('T-20001', 'C-0001', 'delivery');
""", rules)

> **Check.** You should see `Accepted.`

> **Predict.** What happens when you insert T-20001 a second time? Then run the cell.

In [ ]:
try_sql("INSERT INTO tickets VALUES ('T-20001', 'C-0001', 'delivery');", rules)

> **Check.** You should see `Error: UNIQUE constraint failed: tickets.ticket_id`. The database refused the second T-20001.

Now insert a ticket for customer C-0009, which is not in `customers`. Run the cell.

In [ ]:
try_sql("INSERT INTO tickets VALUES ('T-20002', 'C-0009', 'returns');", rules)

> **Check.** You should see `Error: FOREIGN KEY constraint failed`.

> **Predict.** A guest ticket has no customer. Does the foreign key accept NULL? Run the cell.

In [ ]:
try_sql("INSERT INTO tickets VALUES ('T-20002', NULL, 'returns');", rules)
sql("SELECT * FROM tickets;", db=rules)

> **Check.** You should see `Accepted.` and 2 rows. T-20002 has `NULL` as its `customer_id`. A foreign key allows NULL.

The next cell makes the same tables in a second new database, `loose`, without the first line `PRAGMA foreign_keys = ON;`. Then it inserts the ticket for C-0009 again.

> **Predict.** Is the bad row accepted this time? Run the cell.

In [ ]:
loose = sqlite3.connect(":memory:")
try_sql("""
CREATE TABLE customers (
    customer_id TEXT NOT NULL PRIMARY KEY,
    segment     TEXT
);
CREATE TABLE tickets (
    ticket_id   TEXT NOT NULL PRIMARY KEY,
    customer_id TEXT,
    team        TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);
INSERT INTO customers VALUES ('C-0001', 'home'), ('C-0002', 'trade');
INSERT INTO tickets VALUES ('T-20001', 'C-0001', 'delivery');
INSERT INTO tickets VALUES ('T-20002', 'C-0009', 'returns');
""", loose)

> **Check.** You should see `Accepted.` **SQLite checks foreign keys only after `PRAGMA foreign_keys = ON;`**, and you must run it again on every new connection. Without it, `FOREIGN KEY` is only a note in the schema. PostgreSQL always checks foreign keys.

> **Warning.** SQLite accepts NULL in a `PRIMARY KEY` column of type `TEXT`, because of an old design decision. Write `NOT NULL PRIMARY KEY`, as above.

## 5. Why the export has no constraints

The practice database has types, but **no** `PRIMARY KEY` and **no** `FOREIGN KEY` constraints, on purpose. The export does not guarantee keys. With constraints, the first bad row would stop the load, and Mei would have no data to inspect. So she loads every row, then checks the keys herself:

1. **Is the primary key unique?** Compare the number of rows with the number of different values.
2. **Does each foreign key point to a row?** Look for values that are not in the other table. A row whose foreign key points to nothing is an **orphan**.

## 6. Worked example: check `ticket_id`

`COUNT(DISTINCT ticket_id)` counts the different values. The next queries use the real data, `con`.

> **Predict.** Are the two numbers equal? Then run the cell.

In [ ]:
sql("""
SELECT COUNT(*) AS ticket_rows,
       COUNT(DISTINCT ticket_id) AS different_ids
FROM tickets;
""")

> **Check.** You should see **1417** and **1403**. Some IDs appear more than once.

Mei lists them. Read the query as "for each ticket ID, count its rows, and keep the IDs with more than one row". Module 2 teaches `GROUP BY` and `HAVING`. Run the cell.

In [ ]:
sql("""
SELECT ticket_id, COUNT(*) AS row_count
FROM tickets
GROUP BY ticket_id
HAVING COUNT(*) > 1;
""")

> **Check.** You should see 14 rows, from T-10031 to T-11291, each with `row_count` 2.

Compare the two rows of T-10031. Run the cell.

In [ ]:
sql("""
SELECT *
FROM tickets
WHERE ticket_id = 'T-10031';
""")

> **Check.** You should see 2 rows that are the same in all 11 columns: **exact duplicate rows**. Grace explains that two pages of the export overlapped.

`SELECT DISTINCT *` keeps one copy of each different row. If every duplicate is an exact copy, the count of different rows equals the count of different IDs.

> **Predict.** What number do you expect? Run the cell.

In [ ]:
sql("SELECT COUNT(*) AS different_rows FROM (SELECT DISTINCT * FROM tickets);")

> **Check.** You should see **1403**, the same as the number of different IDs. All 14 are exact copies. Mei does not delete them now: Module 4 removes them in a cleaning step that records what it removed. For now: 1,417 rows, 1,403 tickets.

## 7. Worked example: orphan tickets

Next, the foreign key `tickets.customer_id`. `NOT IN (SELECT customer_id FROM customers)` means "not in the list of customer IDs". Run the cell.

In [ ]:
sql("""
SELECT ticket_id, customer_id
FROM tickets
WHERE customer_id NOT IN (SELECT customer_id FROM customers);
""")

> **Check.** You should see 9 rows, with the customer IDs C-0241 to C-0244: deleted accounts.

> **Predict.** The export also has 12 guest tickets, with NULL in `customer_id`. Why are they not in the result? Then run the cell.

In [ ]:
sql("""
SELECT ticket_id, customer_id
FROM tickets
WHERE customer_id IS NULL;
""")

> **Check.** You should see 12 rows, all with `NULL` as the `customer_id`. A comparison with NULL is never true, so `NULL NOT IN (...)` does not keep the row. A check for orphans must also count the NULLs with `IS NULL`. So 21 ticket rows have no matching customer: 9 deleted accounts and 12 guests.

## 8. Guided practice: is `customer_id` a primary key in `customers`?

`COUNT(column)` counts only the values that are not NULL.

> **Your turn.** The starting query compares the number of rows with the number of different customer IDs. Add a third column: `COUNT(customer_id) AS not_null`. Run the cell, then run the check cell.

In [ ]:
customer_key = """
SELECT COUNT(*) AS customer_rows,
       COUNT(DISTINCT customer_id) AS different_ids
FROM customers;
"""
sql(customer_key)

In [ ]:
check(customer_key, rows=1, columns=3, expected="5955459252fcea2f")

> **Check.** You should see `240`, `240` and `240`. So `customer_id` is unique and never NULL: it works as a primary key.

## 9. Your turn: orphan outcome events

> **Your turn.** Each outcome event must belong to a ticket. Write a query that shows every column of the events whose `ticket_id` is not in `tickets`. Use the orphan-tickets query above as a pattern. Predict first: will the result be empty? Then run the check cell.

In [ ]:
orphan_events = """
-- Write your query here
"""
sql(orphan_events)

In [ ]:
check(orphan_events, rows=4, columns=6, expected="ccf843dd301b9d6e")

## 10. Independent variation: is `outcome_id` unique?

> **Your turn.** Check the primary key of `outcomes` as you checked `ticket_id`. Write one query that returns one row with two numbers: the number of rows and the number of different `outcome_id` values. Then run the check cell.

In [ ]:
outcome_key = """
-- Write your query here
"""
sql(outcome_key)

In [ ]:
check(outcome_key, rows=1, columns=2, expected="4baf48e754c54c50")

Both numbers are 1,874, so `outcome_id` is unique. A unique key does not prove that the data has no repeats: Module 3 shows events that were logged twice with a new `outcome_id`.

## 11. Failure case: trusting a key without checking it

**Symptom:** Tomás counts tickets with `COUNT(*)` on `tickets` and gets 1,417. Grace's help-desk screen says 1,403. Nobody saw an error.

**Cause:** Tomás assumed that `ticket_id` is a primary key, because its name ends in `_id`. The export has no constraints, and 14 tickets appear twice.

**Fix:** before you use a column as a key, compare `COUNT(*)` with `COUNT(DISTINCT key)`. If they differ, list the repeated values and look at the rows. Then decide with the people who know the data, and write the decision down.

**In AI work:** duplicated rows can land in the training set and in the test set at the same time. The model is then tested on rows that it has already seen, and the score looks better than it is.

## Check your understanding and recap

Answer the three **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that a **primary key** identifies one row (unique and never NULL), and that a **foreign key** points to the primary key of another table (it can repeat and be NULL). The export has no constraints, so you check keys yourself: `COUNT(*)` against `COUNT(DISTINCT key)`, `NOT IN` for orphans, and `IS NULL` for missing values. The real data has 1,417 ticket rows but 1,403 tickets, 9 tickets of deleted customers, 12 guest tickets and 4 events without a ticket.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.